## はじめに

このノートブックでは、GlacierStyle ECサイトの仕入先商品データと商品マスタの **名寄せ（エンティティ解決）** を体験します。

**名寄せとは:**
- 異なるデータソース間で「同じ実体を指すレコード」を特定・紐づける処理
- 表記揺れ、略称、同義語などにより、単純な完全一致では紐づけできないケースが多い
- データ統合・分析の前処理として非常に重要

**シナリオ:**
- 仕入先5社から届いた商品リスト（`supplier_products`）を、自社の商品マスタ（`dim_products`）に紐づけたい
- しかし仕入先の商品名は表記揺れがあり、完全一致では突合できない

**主な処理内容:**
- 手法0: 正規化（UPPER/TRIM/REPLACE等）による前処理
- 手法1: JAROWINKLER_SIMILARITY による文字列類似度マッチング
- 手法2: AI_SIMILARITY によるAI意味類似度マッチング
- 手法3: EMBED_TEXT + VECTOR_COSINE_SIMILARITY によるベクトル検索
- 手法4: AI_COMPLETE によるLLM判定
- 各手法の比較と最終マッピングテーブルの作成

**前提条件:**
- part1_data_ingest.ipynb が実行済みであること（`dim_products` テーブルが存在すること）

In [ ]:
%%sql -r result_env_setup
-- ============================================================================
-- 環境設定
-- ============================================================================
-- 使用するウェアハウスとスキーマを設定
USE WAREHOUSE GLACIERSTYLE_WH;
USE SCHEMA GLACIERSTYLE_DB.EC_ANALYTICS_SCHEMA;

## 1. データ準備

仕入先から届いた商品リストをテーブルとして読み込みます。
このデータには意図的に表記揺れが含まれています（実際のビジネスでもよくあるケースです）。

**表記揺れの例:**
- 全角/半角の違い: `LED` → `ＬＥＤ`
- スペースの有無: `スタイリッシュデスクライト` → `スタイリッシュ デスクライト`
- 同義語: `ライト` → `ランプ` / `ソファ` → `ソファー`
- 文字の脱落: `北欧風フロアライト` → `北欧フロアライト`
- ブランド名付き: `GlacierStyle スタイリッシュデスクライト`
- カタカナ表記揺れ: `Tシャツ` → `ティーシャツ`

In [ ]:
%%sql -r result_create_supplier
-- ============================================================================
-- 仕入先商品テーブルの作成
-- ============================================================================
CREATE OR REPLACE TABLE supplier_products (
    supplier_product_id VARCHAR PRIMARY KEY,
    supplier_product_name VARCHAR,
    supplier_name VARCHAR,
    supplier_price DECIMAL(10,2),
    supplier_category VARCHAR,
    original_product_id VARCHAR  -- 正解データ（検証用）
);

-- CSVデータの読み込み
COPY INTO supplier_products
FROM @DATA_STAGE/supplier_products.csv
FILE_FORMAT = (
    TYPE = 'CSV'
    SKIP_HEADER = 1
    FIELD_OPTIONALLY_ENCLOSED_BY = '"'
)
ON_ERROR = 'CONTINUE';

In [ ]:
%%sql -r result_preview_supplier
-- ============================================================================
-- 仕入先データのプレビュー
-- ============================================================================
SELECT 
    supplier_product_id,
    supplier_product_name,
    supplier_name,
    supplier_price,
    supplier_category
FROM supplier_products
LIMIT 10;

In [ ]:
%%sql -r result_preview_products
-- ============================================================================
-- 商品マスタのプレビュー（比較対象）
-- ============================================================================
SELECT 
    product_id,
    product_name,
    category_l2,
    brand,
    current_price
FROM dim_products
LIMIT 10;

### 1-1. 完全一致JOINの確認

まず、単純な完全一致JOINでどの程度マッチするか確認しましょう。

In [ ]:
%%sql -r result_exact_match
-- ============================================================================
-- 完全一致JOINの結果確認
-- ============================================================================
-- まず完全一致でどの程度マッチするか確認
WITH exact_match AS (
    SELECT 
        sp.supplier_product_id,
        sp.supplier_product_name,
        dp.product_id,
        dp.product_name
    FROM supplier_products sp
    LEFT JOIN dim_products dp
        ON sp.supplier_product_name = dp.product_name
)
SELECT 
    COUNT(*) AS total_supplier_products,
    COUNT(product_id) AS exact_matched,
    COUNT(*) - COUNT(product_id) AS unmatched,
    ROUND(COUNT(product_id) / COUNT(*) * 100, 1) AS match_rate_pct
FROM exact_match;

In [ ]:
%%sql -r result_unmatched
-- ============================================================================
-- マッチしなかったレコードの例を確認
-- ============================================================================
SELECT 
    sp.supplier_product_name AS 仕入先商品名,
    dp.product_name AS マスタ商品名
FROM supplier_products sp
LEFT JOIN dim_products dp
    ON sp.supplier_product_name = dp.product_name
WHERE dp.product_id IS NULL
LIMIT 15;

## 2. 手法0: 正規化による前処理

AI関数を使う前に、まず基本的な文字列正規化でどの程度改善できるか確認しましょう。

**正規化処理:**
- 全角英数字 → 半角英数字
- 前後の空白除去（TRIM）
- 連続スペースの除去
- 大文字/小文字の統一（UPPER）

In [ ]:
%%sql -r result_normalize_func
-- ============================================================================
-- 正規化用UDF（全角→半角変換 + スペース除去 + 大文字統一）
-- ============================================================================
CREATE OR REPLACE FUNCTION normalize_product_name(name VARCHAR)
RETURNS VARCHAR
LANGUAGE SQL
AS
$$
    UPPER(
        TRIM(
            REGEXP_REPLACE(
                -- 全角英字を半角に変換
                TRANSLATE(
                    TRANSLATE(
                        TRANSLATE(name,
                            'ＡＢＣＤＥＦＧＨＩＪＫＬＭＮＯＰＱＲＳＴＵＶＷＸＹＺａｂｃｄｅｆｇｈｉｊｋｌｍｎｏｐｑｒｓｔｕｖｗｘｙｚ',
                            'ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz'
                        ),
                        -- 全角数字を半角に変換
                        '０１２３４５６７８９',
                        '0123456789'
                    ),
                    -- 全角スペースを半角に変換
                    '　', ' '
                ),
                -- 連続スペースを除去
                '\\s+', ''
            )
        )
    )
$$;

In [ ]:
%%sql -r result_normalize_test
-- ============================================================================
-- 正規化のテスト
-- ============================================================================
SELECT 
    'ＬＥＤ調光デスクライト' AS original,
    normalize_product_name('ＬＥＤ調光デスクライト') AS normalized
UNION ALL
SELECT 
    'スタイリッシュ デスク ライト',
    normalize_product_name('スタイリッシュ デスク ライト')
UNION ALL
SELECT 
    'LED調光デスクライト',
    normalize_product_name('LED調光デスクライト');

In [ ]:
%%sql -r result_normalize_join
-- ============================================================================
-- 正規化後のJOIN結果
-- ============================================================================
WITH normalized_match AS (
    SELECT 
        sp.supplier_product_id,
        sp.supplier_product_name,
        normalize_product_name(sp.supplier_product_name) AS normalized_supplier,
        dp.product_id,
        dp.product_name,
        normalize_product_name(dp.product_name) AS normalized_master
    FROM supplier_products sp
    LEFT JOIN dim_products dp
        ON normalize_product_name(sp.supplier_product_name) = normalize_product_name(dp.product_name)
)
SELECT 
    COUNT(*) AS total_supplier_products,
    COUNT(product_id) AS normalized_matched,
    COUNT(*) - COUNT(product_id) AS still_unmatched,
    ROUND(COUNT(product_id) / COUNT(*) * 100, 1) AS match_rate_pct
FROM normalized_match;

In [ ]:
%%sql -r result_still_unmatched
-- ============================================================================
-- 正規化でもマッチしなかったレコードの例
-- ============================================================================
-- 同義語や文字の脱落は正規化では対応できない
SELECT 
    sp.supplier_product_name AS 仕入先商品名,
    normalize_product_name(sp.supplier_product_name) AS 正規化後
FROM supplier_products sp
LEFT JOIN dim_products dp
    ON normalize_product_name(sp.supplier_product_name) = normalize_product_name(dp.product_name)
WHERE dp.product_id IS NULL
LIMIT 15;

**正規化のまとめ:**
- 全角/半角の違い、スペースの有無などの「表記上の揺れ」は正規化で解決可能
- しかし、同義語（ライト↔ランプ）、文字の脱落（北欧風→北欧）、ブランド名付き等は正規化では対応不可
- → AI関数を活用した高度なマッチングが必要

## 3. 手法1: JAROWINKLER_SIMILARITY（文字列類似度）

Jaro-Winkler類似度は、2つの文字列の編集距離をベースにした類似度指標です。

**特徴:**
- 0〜100の値を返す（100が完全一致）
- 文字の転置、挿入、削除に対して類似度を計算
- 先頭の文字が一致するとボーナス加点（Winkler補正）
- AI関数を使わないため高速・低コスト
- ただし意味的な類似性は考慮しない（ライト≠ランプ）

In [ ]:
%%sql -r result_jw_demo
-- ============================================================================
-- JAROWINKLER_SIMILARITY のデモ
-- ============================================================================
SELECT 
    'スタイリッシュデスクライト' AS name1,
    'スタイリッシュ型デスクライト' AS name2,
    JAROWINKLER_SIMILARITY('スタイリッシュデスクライト', 'スタイリッシュ型デスクライト') AS similarity
UNION ALL
SELECT 
    'モダンデスクライト',
    'モダンデスクランプ',
    JAROWINKLER_SIMILARITY('モダンデスクライト', 'モダンデスクランプ')
UNION ALL
SELECT 
    '北欧風フロアライト',
    '北欧フロアライト',
    JAROWINKLER_SIMILARITY('北欧風フロアライト', '北欧フロアライト')
UNION ALL
SELECT 
    'LED調光デスクライト',
    'エルイーディー調光デスクライト',
    JAROWINKLER_SIMILARITY('LED調光デスクライト', 'エルイーディー調光デスクライト');

In [ ]:
%%sql -r result_jw_match
-- ============================================================================
-- JAROWINKLER_SIMILARITY による名寄せ
-- ============================================================================
-- 正規化済みの名前同士で類似度を計算し、最も類似度の高い候補を取得
CREATE OR REPLACE TABLE work_jarowinkler_match AS
WITH similarity_calc AS (
    SELECT 
        sp.supplier_product_id,
        sp.supplier_product_name,
        dp.product_id,
        dp.product_name,
        JAROWINKLER_SIMILARITY(
            normalize_product_name(sp.supplier_product_name),
            normalize_product_name(dp.product_name)
        ) AS jw_similarity,
        ROW_NUMBER() OVER (
            PARTITION BY sp.supplier_product_id 
            ORDER BY JAROWINKLER_SIMILARITY(
                normalize_product_name(sp.supplier_product_name),
                normalize_product_name(dp.product_name)
            ) DESC
        ) AS rank
    FROM supplier_products sp
    CROSS JOIN dim_products dp
)
SELECT 
    supplier_product_id,
    supplier_product_name,
    product_id AS matched_product_id,
    product_name AS matched_product_name,
    jw_similarity
FROM similarity_calc
WHERE rank = 1;

-- 結果確認
SELECT * FROM work_jarowinkler_match
ORDER BY jw_similarity ASC
LIMIT 20;

In [ ]:
%%sql -r result_jw_accuracy
-- ============================================================================
-- JAROWINKLER_SIMILARITY の精度検証
-- ============================================================================
-- 正解データ（original_product_id）と比較して精度を確認
-- 閾値: 70 で判定
SELECT 
    COUNT(*) AS total,
    SUM(CASE WHEN jw.matched_product_id = sp.original_product_id THEN 1 ELSE 0 END) AS correct_matches,
    SUM(CASE WHEN jw.jw_similarity >= 70 AND jw.matched_product_id = sp.original_product_id THEN 1 ELSE 0 END) AS correct_above_threshold,
    SUM(CASE WHEN jw.jw_similarity >= 70 AND jw.matched_product_id != sp.original_product_id THEN 1 ELSE 0 END) AS incorrect_above_threshold,
    SUM(CASE WHEN jw.jw_similarity < 70 THEN 1 ELSE 0 END) AS below_threshold,
    ROUND(SUM(CASE WHEN jw.matched_product_id = sp.original_product_id THEN 1 ELSE 0 END) / COUNT(*) * 100, 1) AS overall_accuracy_pct
FROM work_jarowinkler_match jw
JOIN supplier_products sp ON jw.supplier_product_id = sp.supplier_product_id;

### ★ ハンズオン: 閾値を変えてみましょう

上のクエリの閾値 `70` を変更して、精度の変化を確認してみましょう。
- 閾値を上げると（例: 80）→ 誤マッチが減るが、見逃しが増える
- 閾値を下げると（例: 60）→ 見逃しが減るが、誤マッチが増える

In [ ]:
%%sql -r result_jw_handson
-- ============================================================================
-- ★ ハンズオン: 閾値を変更して精度を確認
-- ============================================================================
-- <***> 以下の閾値を変更してみましょう（例: 60, 75, 80, 90）
SET threshold = 70;

SELECT 
    $threshold AS 閾値,
    COUNT(*) AS total,
    SUM(CASE WHEN jw.jw_similarity >= $threshold AND jw.matched_product_id = sp.original_product_id THEN 1 ELSE 0 END) AS 正解マッチ数,
    SUM(CASE WHEN jw.jw_similarity >= $threshold AND jw.matched_product_id != sp.original_product_id THEN 1 ELSE 0 END) AS 誤マッチ数,
    SUM(CASE WHEN jw.jw_similarity < $threshold THEN 1 ELSE 0 END) AS 閾値未満数,
    ROUND(
        SUM(CASE WHEN jw.jw_similarity >= $threshold AND jw.matched_product_id = sp.original_product_id THEN 1 ELSE 0 END) 
        / NULLIF(SUM(CASE WHEN jw.jw_similarity >= $threshold THEN 1 ELSE 0 END), 0) * 100, 1
    ) AS 適合率_pct,
    ROUND(
        SUM(CASE WHEN jw.jw_similarity >= $threshold AND jw.matched_product_id = sp.original_product_id THEN 1 ELSE 0 END) 
        / COUNT(*) * 100, 1
    ) AS 再現率_pct
FROM work_jarowinkler_match jw
JOIN supplier_products sp ON jw.supplier_product_id = sp.supplier_product_id;

## 4. 手法2: AI_SIMILARITY（AI意味類似度）

Snowflake Cortex の `AI_SIMILARITY` は、2つのテキストの意味的な類似度を計算します。

**特徴:**
- -1〜1の値を返す（1が完全一致）
- 意味的な類似性を考慮（ライト≒ランプ、ソファ≒カウチ）
- Jaro-Winklerでは対応できない同義語パターンに強い
- AI関数のためコストが発生するが、高精度

**注意:** CROSS JOINは計算量が大きいため、サンプルデータで動作を確認します。

In [ ]:
%%sql -r result_ais_demo
-- ============================================================================
-- AI_SIMILARITY のデモ
-- ============================================================================
SELECT 
    'モダンデスクライト' AS name1,
    'モダンデスクランプ' AS name2,
    AI_SIMILARITY('モダンデスクライト', 'モダンデスクランプ') AS ai_similarity,
    JAROWINKLER_SIMILARITY('モダンデスクライト', 'モダンデスクランプ') AS jw_similarity
UNION ALL
SELECT 
    'プレミアムカシミヤセーター',
    '高級カシミアニット',
    AI_SIMILARITY('プレミアムカシミヤセーター', '高級カシミアニット'),
    JAROWINKLER_SIMILARITY('プレミアムカシミヤセーター', '高級カシミアニット')
UNION ALL
SELECT 
    'ワイヤレスヘッドホン',
    '無線ヘッドフォン',
    AI_SIMILARITY('ワイヤレスヘッドホン', '無線ヘッドフォン'),
    JAROWINKLER_SIMILARITY('ワイヤレスヘッドホン', '無線ヘッドフォン')
UNION ALL
SELECT 
    'オーガニックコットンTシャツ',
    '有機綿ティーシャツ',
    AI_SIMILARITY('オーガニックコットンTシャツ', '有機綿ティーシャツ'),
    JAROWINKLER_SIMILARITY('オーガニックコットンTシャツ', '有機綿ティーシャツ');

In [ ]:
%%sql -r result_ais_match
-- ============================================================================
-- AI_SIMILARITY による名寄せ（サンプル50件）
-- ============================================================================
-- 全件CROSS JOINはコストが高いため、サンプルで実施
CREATE OR REPLACE TABLE work_ai_similarity_match AS
WITH sample_suppliers AS (
    SELECT * FROM supplier_products SAMPLE (50 ROWS)
),
similarity_calc AS (
    SELECT 
        sp.supplier_product_id,
        sp.supplier_product_name,
        dp.product_id,
        dp.product_name,
        AI_SIMILARITY(sp.supplier_product_name, dp.product_name) AS ai_sim,
        ROW_NUMBER() OVER (
            PARTITION BY sp.supplier_product_id 
            ORDER BY AI_SIMILARITY(sp.supplier_product_name, dp.product_name) DESC
        ) AS rank
    FROM sample_suppliers sp
    CROSS JOIN dim_products dp
)
SELECT 
    supplier_product_id,
    supplier_product_name,
    product_id AS matched_product_id,
    product_name AS matched_product_name,
    ai_sim AS ai_similarity
FROM similarity_calc
WHERE rank = 1;

-- 結果確認
SELECT * FROM work_ai_similarity_match
ORDER BY ai_similarity ASC
LIMIT 20;

In [ ]:
%%sql -r result_ais_accuracy
-- ============================================================================
-- AI_SIMILARITY の精度検証
-- ============================================================================
SELECT 
    COUNT(*) AS total,
    SUM(CASE WHEN ais.matched_product_id = sp.original_product_id THEN 1 ELSE 0 END) AS correct_matches,
    ROUND(SUM(CASE WHEN ais.matched_product_id = sp.original_product_id THEN 1 ELSE 0 END) / COUNT(*) * 100, 1) AS accuracy_pct
FROM work_ai_similarity_match ais
JOIN supplier_products sp ON ais.supplier_product_id = sp.supplier_product_id;

## 5. 手法3: EMBED_TEXT + VECTOR_COSINE_SIMILARITY（ベクトル検索）

テキストをベクトル（数値の配列）に変換し、ベクトル間のコサイン類似度で名寄せを行います。

**特徴:**
- テキストの意味を多次元ベクトルとして表現
- 一度ベクトル化すれば、検索は高速（事前計算可能）
- 大量データに対してスケーラブル
- Cortex Search Serviceの内部でも使われている技術

In [ ]:
%%sql -r result_embed_demo
-- ============================================================================
-- EMBED_TEXT のデモ
-- ============================================================================
-- テキストをベクトルに変換
SELECT 
    'モダンデスクライト' AS text,
    SNOWFLAKE.CORTEX.EMBED_TEXT_1024('snowflake-arctic-embed-l-v2.0', 'モダンデスクライト') AS embedding
LIMIT 1;

In [ ]:
%%sql -r result_embed_products
-- ============================================================================
-- 商品マスタの商品名をベクトル化（事前計算）
-- ============================================================================
CREATE OR REPLACE TABLE dim_products_with_embedding AS
SELECT 
    product_id,
    product_name,
    SNOWFLAKE.CORTEX.EMBED_TEXT_1024('snowflake-arctic-embed-l-v2.0', product_name) AS product_name_embedding
FROM dim_products;

In [ ]:
%%sql -r result_vector_match
-- ============================================================================
-- ベクトル類似度による名寄せ（サンプル50件）
-- ============================================================================
CREATE OR REPLACE TABLE work_vector_match AS
WITH sample_suppliers AS (
    SELECT 
        supplier_product_id,
        supplier_product_name,
        SNOWFLAKE.CORTEX.EMBED_TEXT_1024('snowflake-arctic-embed-l-v2.0', supplier_product_name) AS supplier_embedding
    FROM supplier_products SAMPLE (50 ROWS)
),
similarity_calc AS (
    SELECT 
        sp.supplier_product_id,
        sp.supplier_product_name,
        dp.product_id,
        dp.product_name,
        VECTOR_COSINE_SIMILARITY(sp.supplier_embedding, dp.product_name_embedding) AS cosine_similarity,
        ROW_NUMBER() OVER (
            PARTITION BY sp.supplier_product_id 
            ORDER BY VECTOR_COSINE_SIMILARITY(sp.supplier_embedding, dp.product_name_embedding) DESC
        ) AS rank
    FROM sample_suppliers sp
    CROSS JOIN dim_products_with_embedding dp
)
SELECT 
    supplier_product_id,
    supplier_product_name,
    product_id AS matched_product_id,
    product_name AS matched_product_name,
    cosine_similarity
FROM similarity_calc
WHERE rank = 1;

-- 結果確認
SELECT * FROM work_vector_match
ORDER BY cosine_similarity ASC
LIMIT 20;

In [ ]:
%%sql -r result_vector_accuracy
-- ============================================================================
-- ベクトル検索の精度検証
-- ============================================================================
SELECT 
    COUNT(*) AS total,
    SUM(CASE WHEN vm.matched_product_id = sp.original_product_id THEN 1 ELSE 0 END) AS correct_matches,
    ROUND(SUM(CASE WHEN vm.matched_product_id = sp.original_product_id THEN 1 ELSE 0 END) / COUNT(*) * 100, 1) AS accuracy_pct
FROM work_vector_match vm
JOIN supplier_products sp ON vm.supplier_product_id = sp.supplier_product_id;

## 6. 手法4: AI_COMPLETE（LLM判定）

LLM（大規模言語モデル）に直接「この2つの商品名は同じ商品を指すか？」を判定させます。

**特徴:**
- 最も高い精度が期待できる（文脈理解力が高い）
- プロンプトエンジニアリングで精度を調整可能
- ただしコスト・処理時間が最も大きい
- 大量データに対してはコスト面で注意が必要

**実践的なアプローチ:**
- 他の手法で絞り込んだ「判断が難しい候補」に対してLLMを適用するのが効率的

In [ ]:
%%sql -r result_llm_demo
-- ============================================================================
-- AI_COMPLETE によるLLM判定のデモ
-- ============================================================================
SELECT 
    'モダンデスクライト' AS master_name,
    'モダンデスクランプ' AS supplier_name,
    AI_COMPLETE(
        'llama4-maverick',
        '以下の2つの商品名が同じ商品を指すかどうかを判定してください。\n'
        || '回答は必ず「YES」または「NO」のみで答えてください。\n\n'
        || '商品マスタ: モダンデスクライト\n'
        || '仕入先商品: モダンデスクランプ',
        {'temperature': 0}
    ) AS llm_judgment;

In [ ]:
%%sql -r result_llm_match
-- ============================================================================
-- AI_COMPLETE による名寄せ（サンプル20件）
-- ============================================================================
-- JaroWinklerで絞り込んだ上位候補に対してLLM判定を適用
CREATE OR REPLACE TABLE work_llm_match AS
WITH sample_suppliers AS (
    SELECT * FROM supplier_products SAMPLE (20 ROWS)
),
-- まずJaroWinklerで上位3候補に絞り込み
top_candidates AS (
    SELECT 
        sp.supplier_product_id,
        sp.supplier_product_name,
        dp.product_id,
        dp.product_name,
        JAROWINKLER_SIMILARITY(
            normalize_product_name(sp.supplier_product_name),
            normalize_product_name(dp.product_name)
        ) AS jw_sim,
        ROW_NUMBER() OVER (
            PARTITION BY sp.supplier_product_id 
            ORDER BY JAROWINKLER_SIMILARITY(
                normalize_product_name(sp.supplier_product_name),
                normalize_product_name(dp.product_name)
            ) DESC
        ) AS rank
    FROM sample_suppliers sp
    CROSS JOIN dim_products dp
),
-- 上位3候補に対してLLM判定
llm_judged AS (
    SELECT 
        *,
        AI_COMPLETE(
            'llama4-maverick',
            '以下の2つの商品名が同じ商品を指すかどうかを判定してください。\n'
            || '回答は必ず「YES」または「NO」のみで答えてください。\n\n'
            || '商品マスタ: ' || product_name || '\n'
            || '仕入先商品: ' || supplier_product_name,
            {'temperature': 0}
        ) AS llm_result
    FROM top_candidates
    WHERE rank <= 3
)
SELECT 
    supplier_product_id,
    supplier_product_name,
    product_id AS matched_product_id,
    product_name AS matched_product_name,
    jw_sim,
    llm_result,
    CASE WHEN CONTAINS(UPPER(llm_result), 'YES') THEN TRUE ELSE FALSE END AS is_match
FROM llm_judged
WHERE CONTAINS(UPPER(llm_result), 'YES')
QUALIFY ROW_NUMBER() OVER (PARTITION BY supplier_product_id ORDER BY jw_sim DESC) = 1;

-- 結果確認
SELECT * FROM work_llm_match;

In [ ]:
%%sql -r result_llm_accuracy
-- ============================================================================
-- LLM判定の精度検証
-- ============================================================================
SELECT 
    COUNT(*) AS total,
    SUM(CASE WHEN lm.matched_product_id = sp.original_product_id THEN 1 ELSE 0 END) AS correct_matches,
    ROUND(SUM(CASE WHEN lm.matched_product_id = sp.original_product_id THEN 1 ELSE 0 END) / COUNT(*) * 100, 1) AS accuracy_pct
FROM work_llm_match lm
JOIN supplier_products sp ON lm.supplier_product_id = sp.supplier_product_id;

### ★ ハンズオン: プロンプトを改善してみましょう

上のLLM判定のプロンプトを変更して、精度の変化を確認してみましょう。

例:
- カテゴリ情報を追加する
- 具体的な判断基準を与える（「略称や同義語は同じ商品とみなす」など）
- 出力形式をJSON形式にする

In [ ]:
%%sql -r result_llm_handson
-- ============================================================================
-- ★ ハンズオン: プロンプトを改善してみましょう
-- ============================================================================
-- <***> 以下のプロンプトを改善してみましょう
SELECT 
    'オーガニックコットンTシャツ' AS master_name,
    '有機綿ティーシャツ' AS supplier_name,
    AI_COMPLETE(
        'llama4-maverick',
        '以下の2つの商品名が同じ商品を指すかどうかを判定してください。\n'
        || '同義語、略称、表記揺れ（カタカナ/英語、全角/半角など）は同じ商品とみなしてください。\n'
        || '回答は必ず「YES」または「NO」のみで答えてください。\n\n'
        || '商品マスタ: オーガニックコットンTシャツ\n'
        || '仕入先商品: 有機綿ティーシャツ',
        {'temperature': 0}
    ) AS llm_judgment;

## 7. 手法比較とまとめ

ここまでに試した5つの手法の特徴を比較します。

**手法の段階的な活用:**

1. **正規化**（コスト: ゼロ）→ 表記上の揺れを解消
2. **JAROWINKLER_SIMILARITY**（コスト: 低）→ 文字レベルの類似度で候補を絞り込み
3. **AI_SIMILARITY / EMBED_TEXT**（コスト: 中）→ 意味的な類似度で精度向上
4. **AI_COMPLETE**（コスト: 高）→ 判断が難しいケースをLLMで最終判定

**実践的なアプローチ:**
- まず正規化で解決可能なものを処理
- 残りをJaroWinklerで絞り込み
- 閾値付近の「グレーゾーン」をAI関数で最終判定
- → コストと精度のバランスを最適化

In [ ]:
%%sql -r result_final_mapping
-- ============================================================================
-- 最終マッピングテーブルの作成
-- ============================================================================
-- 段階的アプローチ:
-- Step 1: 正規化で完全一致するものを先にマッチ
-- Step 2: 残りをJaroWinkler + AI_SIMILARITYで候補生成
CREATE OR REPLACE TABLE gold_supplier_product_mapping AS

-- Step 1: 正規化による完全一致
WITH normalized_match AS (
    SELECT 
        sp.supplier_product_id,
        sp.supplier_product_name,
        sp.supplier_name,
        dp.product_id AS matched_product_id,
        dp.product_name AS matched_product_name,
        1.0 AS match_score,
        '正規化完全一致' AS match_method
    FROM supplier_products sp
    JOIN dim_products dp
        ON normalize_product_name(sp.supplier_product_name) = normalize_product_name(dp.product_name)
    QUALIFY ROW_NUMBER() OVER (PARTITION BY sp.supplier_product_id ORDER BY dp.product_id) = 1
),

-- Step 2: 正規化でマッチしなかったものをJaroWinklerで候補生成
remaining AS (
    SELECT sp.*
    FROM supplier_products sp
    LEFT JOIN normalized_match nm ON sp.supplier_product_id = nm.supplier_product_id
    WHERE nm.supplier_product_id IS NULL
),
jw_match AS (
    SELECT 
        sp.supplier_product_id,
        sp.supplier_product_name,
        sp.supplier_name,
        dp.product_id AS matched_product_id,
        dp.product_name AS matched_product_name,
        JAROWINKLER_SIMILARITY(
            normalize_product_name(sp.supplier_product_name),
            normalize_product_name(dp.product_name)
        ) / 100.0 AS match_score,
        'JaroWinkler類似度' AS match_method
    FROM remaining sp
    CROSS JOIN dim_products dp
    QUALIFY ROW_NUMBER() OVER (
        PARTITION BY sp.supplier_product_id 
        ORDER BY JAROWINKLER_SIMILARITY(
            normalize_product_name(sp.supplier_product_name),
            normalize_product_name(dp.product_name)
        ) DESC
    ) = 1
)

-- 結合
SELECT * FROM normalized_match
UNION ALL
SELECT * FROM jw_match;

-- 結果確認
SELECT match_method, COUNT(*) AS count, ROUND(AVG(match_score), 3) AS avg_score
FROM gold_supplier_product_mapping
GROUP BY match_method;

In [ ]:
%%sql -r result_final_accuracy
-- ============================================================================
-- 最終マッピングの精度確認
-- ============================================================================
SELECT 
    gm.match_method,
    COUNT(*) AS total,
    SUM(CASE WHEN gm.matched_product_id = sp.original_product_id THEN 1 ELSE 0 END) AS correct,
    ROUND(SUM(CASE WHEN gm.matched_product_id = sp.original_product_id THEN 1 ELSE 0 END) / COUNT(*) * 100, 1) AS accuracy_pct
FROM gold_supplier_product_mapping gm
JOIN supplier_products sp ON gm.supplier_product_id = sp.supplier_product_id
GROUP BY gm.match_method

UNION ALL

SELECT 
    '全体' AS match_method,
    COUNT(*) AS total,
    SUM(CASE WHEN gm.matched_product_id = sp.original_product_id THEN 1 ELSE 0 END) AS correct,
    ROUND(SUM(CASE WHEN gm.matched_product_id = sp.original_product_id THEN 1 ELSE 0 END) / COUNT(*) * 100, 1) AS accuracy_pct
FROM gold_supplier_product_mapping gm
JOIN supplier_products sp ON gm.supplier_product_id = sp.supplier_product_id;

In [ ]:
%%sql -r result_preview_final
-- ============================================================================
-- 最終マッピング結果の確認
-- ============================================================================
SELECT 
    gm.supplier_product_id,
    gm.supplier_product_name,
    gm.matched_product_name,
    gm.match_score,
    gm.match_method,
    CASE WHEN gm.matched_product_id = sp.original_product_id THEN '✓ 正解' ELSE '✗ 不正解' END AS verification
FROM gold_supplier_product_mapping gm
JOIN supplier_products sp ON gm.supplier_product_id = sp.supplier_product_id
ORDER BY gm.match_method, gm.match_score DESC
LIMIT 30;

## まとめ

本パートでは、名寄せ（エンティティ解決）の5つの手法を学びました:

1. **正規化**: コストゼロで表記上の揺れを解消。まず最初に適用すべき
2. **JAROWINKLER_SIMILARITY**: 高速・低コストな文字列類似度。大量データの一次スクリーニングに最適
3. **AI_SIMILARITY**: 意味的な類似度を考慮。同義語に強い
4. **EMBED_TEXT + VECTOR_COSINE_SIMILARITY**: ベクトル検索。事前計算でスケーラブル
5. **AI_COMPLETE**: LLMによる最終判定。最高精度だがコスト高

**実践のポイント:**
- 段階的に手法を適用し、コストと精度のバランスを取る
- 正規化→文字列類似度→AI関数の順にフィルタリング
- 「グレーゾーン」にのみ高コストなLLMを適用

**次のパート:**
- Part 3（データ加工・変換）では、SNS投稿からAIが抽出した商品名と商品マスタを突合する処理で、ここで学んだ `AI_SIMILARITY` を実践的に活用します。